# Exploratory Data Analysis: Nationwide Retail Demand & Hierarchy
**Project:** Nationwide Demand Forecasting & Inventory Optimization  
**Dataset:** Walmart M5 Forecasting Data  

### Objectives:
1. Evaluate nationwide demand trends and temporal seasonality (weekly cycles, monthly shifts).
2. Quantify holiday and SNAP promotional demand lifts across California, Texas, and Wisconsin.
3. Analyze retail hierarchy distributions: State, Store, Department, Category, and Item.
4. Assess zero-sales frequency and demand intermittency.
5. Examine sell price variance and price elasticity indicators.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
import sys
sys.path.append('..')
from config.settings import CALENDAR_PATH, SALES_TRAIN_VAL_PATH, PRICES_PATH, SAMPLE_DATA_DIR
from src.data.loader import load_calendar, load_sell_prices, load_raw_sales
from src.data.validation import validate_sales_data, detect_intermittent_demand

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
print('Dependencies loaded successfully!')

## 1. Calendar Events & SNAP Benefit Distribution

In [ ]:
calendar = load_calendar(CALENDAR_PATH)
print('Calendar shape:', calendar.shape)
print('Unique Event Types:', calendar['event_type_1'].value_counts())

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
calendar['event_type_1'].value_counts().plot(kind='bar', ax=ax[0], color='royalblue', title='Calendar Event Distribution')
ax[0].set_ylabel('Number of Days')

snap_counts = pd.DataFrame({
    'CA': calendar['snap_CA'].value_counts(),
    'TX': calendar['snap_TX'].value_counts(),
    'WI': calendar['snap_WI'].value_counts()
})
snap_counts.plot(kind='bar', ax=ax[1], title='SNAP Eligible Days (0 = Inactive, 1 = Active)')
plt.tight_layout()
plt.show()

## 2. Store & State Aggregate Demand Patterns

In [ ]:
sales_raw = load_raw_sales(SALES_TRAIN_VAL_PATH)
d_cols = [c for c in sales_raw.columns if c.startswith('d_')]
state_totals = sales_raw.groupby('state_id')[d_cols].sum().T
state_totals.index = pd.to_datetime(calendar.set_index('d').loc[state_totals.index, 'date'])

state_totals.rolling(28).mean().plot(title='28-Day Rolling Average Demand by State (CA, TX, WI)')
plt.ylabel('Daily Units Sold')
plt.xlabel('Date')
plt.show()

## 3. Department & Category Demand Breakdown

In [ ]:
dept_totals = sales_raw.groupby('dept_id')[d_cols].sum().sum(axis=1).sort_values(ascending=False)
cat_totals = sales_raw.groupby('cat_id')[d_cols].sum().sum(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
cat_totals.plot(kind='pie', autopct='%1.1f%%', ax=ax[0], title='Total Sales Share by Category')
dept_totals.plot(kind='barh', ax=ax[1], color='teal', title='Total Sales Volume by Department')
ax[1].set_xlabel('Total Units Sold')
plt.tight_layout()
plt.show()

## 4. Zero-Sales Frequency & Intermittency Analysis

In [ ]:
# Sample 100 random items to compute intermittency distribution
sample_series = sales_raw.sample(n=100, random_state=42)
zero_rates = (sample_series[d_cols] == 0).mean(axis=1)

plt.figure(figsize=(8, 4))
sns.histplot(zero_rates, bins=20, kde=True, color='crimson')
plt.title('Distribution of Zero-Sales Proportion across Series')
plt.xlabel('Zero Sales Rate (0.0 to 1.0)')
plt.ylabel('Item Series Count')
plt.axvline(zero_rates.median(), color='black', linestyle='--', label=f'Median Zero Rate: {zero_rates.median():.2f}')
plt.legend()
plt.show()

### EDA Findings & Implications for Inventory Optimization
1. **High Intermittency**: Many retail series show zero sales on >40% of days. Croston or Tweedie/Poisson objectives in LightGBM and quantile regression in TFT handle zero-inflation better than standard Gaussian assumptions.
2. **Strong Weekly Seasonality**: Saturday and Sunday demand consistently surges across all states, demanding 7-day cyclical features and safety stock buffers ahead of weekends.
3. **State Dominance**: California (CA) accounts for the largest share of national sales volume, followed by Texas (TX) and Wisconsin (WI).
4. **Price Variations**: Sell price changes generally occur on week boundaries, reinforcing the weekly price lag and percentage discount feature design.